This creates the data used in the tutorials.

In [1]:
import numpy as np
import plopp as pp
import scipp as sc

import easydynamics.sample_model as sm
from easydynamics.sample_model import ComponentCollection
from easydynamics.sample_model import Gaussian
from easydynamics.sample_model.sample_model import SampleModel

%matplotlib widget

In [2]:
# Create data for tutorial0_basics.ipynb. Data is called fake_simple_data.hdf5.

Q = sc.linspace(start=0.2, stop=2.1, num=32, unit='1/angstrom', dim='Q')

component_collection = ComponentCollection()
component_collection.append_component(Gaussian(area=0.45, width=0.1))

model = SampleModel(components=component_collection, Q=Q)

for i in range(Q.size):
    components = model.get_component_collection(i)
    offset = 0.0
    components.components[0].area = 3.79 - 0.2 * Q[i].value

energy = sc.linspace(start=-3.0, stop=3.0, num=756, unit='meV', dim='energy')

intensity = sc.array(values=model.evaluate(x=energy), dims=['Q', 'energy'])

intensity_dataarray = sc.DataArray(data=intensity, coords={'Q': Q, 'energy': energy})

rng = np.random.default_rng()
noise = rng.normal(loc=0.0, scale=1.55, size=intensity_dataarray.shape)
intensity_dataarray.values += noise
intensity_dataarray.variances = noise**2

sc.io.save_hdf5(intensity_dataarray, 'fake_simple_data.hdf5')
pp.slicer(intensity_dataarray)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [3]:
# Create data for tutorial0_more_advanced.ipynb. Data is called fake_advanced_data.hdf5.
Q = sc.linspace(start=0.2, stop=2.1, num=32, unit='1/angstrom', dim='Q')

component_collection = sm.ComponentCollection()
component_collection.append_component(sm.Gaussian(area=0.45, width=0.05))
component_collection.append_component(sm.Lorentzian(area=10.45, width=0.4))
component_collection.append_component(
    sm.DampedHarmonicOscillator(area=1.45, width=0.1, center=1.4)
)
component_collection.append_component(sm.Polynomial(coefficients=[1.2, 0.2]))
model = sm.SampleModel(components=component_collection, Q=Q)


energy = sc.linspace(start=-3.0, stop=3.0, num=756, unit='meV', dim='energy')
intensity_values = np.zeros((Q.size, energy.size))
rng = np.random.default_rng()
noise = rng.normal(loc=0.0, scale=0.35, size=intensity_dataarray.shape)

for i in range(Q.size):
    components = model.get_component_collection(i)
    offset = sc.scalar(value=rng.uniform(0.05, 0.15), unit='meV')
    components.components[0].area = 3.79 - 0.2 * Q[i].value
    components.components[2].center = 1.4 + Q[i].value / 10.0
    components.components[2].area = 2.45 + Q[i].value / 10.0

    intensity_values[i, :] = components.evaluate(x=energy.values - offset.value)


intensity = sc.array(values=intensity_values, dims=['Q', 'energy'])
intensity_dataarray = sc.DataArray(data=intensity, coords={'Q': Q, 'energy': energy})

noise = rng.normal(loc=0.0, scale=0.35, size=intensity_dataarray.shape)
intensity_dataarray.values += noise
intensity_dataarray.variances = noise**2

sc.io.save_hdf5(intensity_dataarray, 'fake_advanced_data.hdf5')
pp.slicer(intensity_dataarray)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…